Finding Internal Audit Data tables

In [0]:
%sql
select distinct 
  upper(trim(Audit_BO.BU)) as BU, 
  upper(trim(Audit_BO.table_name)) as BO_used_table,
  upper(trim(v3_source.SOURCE_TABLE)) as Source_table,
  upper(trim(v3_source.source)) as Source 
from (
  select distinct 
    UP.BU, 
    UA.WEBI_Doc_ID, 
    UA.Audit_id as CUID, 
    UA.Audit_name as Report_Name, 
    sql.tableName as table_name
  from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as UA
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_user_profile as UP
    on UA.UserName = UP.UserName
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.bo_doc_sql_tables as sql
    on UA.audit_id = sql.documentCUID
  where
    upper(UP.BU)="AUDIT-INTERNAL AUDIT" 
    and sql.tableName is not null
) as Audit_BO
left join (
  SELECT DISTINCT
    table as BO_TABLE,
    CASE 
      WHEN upper(SOURCE) LIKE 'ALIAS%' AND base_table IS NOT NULL AND base_table != '' 
      THEN base_table 
      ELSE TABLE 
    END AS SOURCE_TABLE,
    SCHEMA,
    CASE
      WHEN upper(SOURCE) LIKE 'ALIAS%'
      THEN trim(regexp_replace(regexp_replace(SOURCE, '(?i)^Alias\\s*', ''), '[()\\[\\]{}]', ''))
      ELSE SOURCE
    END AS SOURCE
  FROM dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.v3_extraction
  UNION
  SELECT DISTINCT
    table as BO_TABLE,
    CASE 
      WHEN upper(SOURCE) LIKE 'ALIAS%' AND base_table IS NOT NULL AND base_table != '' 
      THEN base_table 
      ELSE TABLE 
    END AS SOURCE_TABLE,
    SCHEMA,
    CASE
      WHEN upper(SOURCE) LIKE 'ALIAS%'
      THEN trim(regexp_replace(regexp_replace(SOURCE, '(?i)^Alias\\s*', ''), '[()\\[\\]{}]', ''))
      ELSE SOURCE
    END AS SOURCE
  FROM dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.v3_extraction_remaining
) v3_source
on upper(trim(Audit_BO.table_name)) = upper(trim(v3_source.BO_TABLE))
order by BO_used_table asc

Test GITHUB

In [0]:
%sql
SELECT DISTINCT
  CASE 
    WHEN upper(SOURCE) LIKE 'ALIAS%' AND base_table IS NOT NULL AND base_table != '' 
    THEN base_table 
    ELSE TABLE 
  END AS TABLE,
  SCHEMA,
  CASE
    WHEN upper(SOURCE) LIKE 'ALIAS%'
    THEN trim(regexp_replace(regexp_replace(SOURCE, '(?i)^Alias\\s*', ''), '[()\\[\\]{}]', ''))
    ELSE SOURCE
  END AS SOURCE
FROM dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.v3_extraction as v3_extraction
where 
upper(v3_extraction.table) ="A_COUNTRIES_REGS" 
UNION
SELECT DISTINCT
  CASE 
    WHEN upper(SOURCE) LIKE 'ALIAS%' AND base_table IS NOT NULL AND base_table != '' 
    THEN base_table 
    ELSE TABLE 
  END AS TABLE,
  SCHEMA,
  CASE
    WHEN upper(SOURCE) LIKE 'ALIAS%'
    THEN trim(regexp_replace(regexp_replace(SOURCE, '(?i)^Alias\\s*', ''), '[()\\[\\]{}]', ''))
    ELSE SOURCE
  END AS SOURCE
FROM dataplatform01_modelling_dev_catalog_01.bo_universes_excel_sheet_imports.v3_extraction_remaining as v3_extraction_remaining
where upper(v3_extraction_remaining.table) ="A_COUNTRIES_REGS" 